# COMPSCI 714 - Lectutorial 5 - Learning with images - CNNs & ViT

Some of the content in this notebook is reused or adapted from [Aurelien Geron repository](https://github.com/ageron).

## Coding time 1 - Convolutional Neural Networks (CNNs)

In [3]:
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from sklearn.datasets import load_sample_images

from functools import partial
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import torchvision
import torchvision.transforms.v2 as T

from torchsummary import summary

In [2]:
%pip install torchmetrics
import torchmetrics

Note: you may need to restart the kernel to use updated packages.


c:\Users\keanu\anaconda3\envs\compsci714\lib\site-packages\transformers\utils\generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


It is recommended to run this Notebook on a GPU (e.g., on Colab) to accelarate the training of the CNN model.

In [4]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

### CNN layers

#### 1. Convolution layer

Convolutional layers are the basis of *Convolutional Neural Networks* (CNNs). In a CNN, convolutional layers take an image or feature map as input, and output a feature map (i.e., a filtered or compressed version of the input image). One important difference between convolutional layers and fully-connected layers is the way they connect inputs and outputs.
In a fully-connected layer, each input is connected to each output. In other words, every neuron in one layer is connected to every single neuron in the next layer.
In a convolutional layer, each input (i.e., pixel) is connected only to a specific *field* of outputs, defined by the filter size associated to this layer. In other words, each neuron in a layer is only connected to neurons located in a small rectangle of the previous layer.

We will first apply a convolutional layer on an image to see what effect it has on sample images.

In [ ]:
sample_images = load_sample_images()["images"]
sample_images = torch.tensor(sample_images, dtype=torch.float32) / 255

Let's have a quick look at the shape of the sample images.

In [ ]:
sample_images.shape

**TODO**: Can you display each image?

In [ ]:
# TODO

PyTorch expects the channels dimension just before the height and width dimensions. We need to permute the dimensions:

In [ ]:
sample_images = sample_images.permute(0, 3, 1, 2)
sample_images.shape

Let's reduce the size of the images by center cropping them:

In [ ]:
cropped_images = T.CenterCrop((70, 120))(sample_images)
cropped_images.shape

In [ ]:
plt.imshow(cropped_images[0].permute(1, 2, 0))

Let's now create a convolution layer and apply it to the images. We can directly use the `nn.Conv2` layer provided in PyTorch. We are using 32 filters of size $7 \times 7 \times 3$.

**TODO**: Why do we set filters with 3 channels? Calculate the expected shape of the output feature map.

In [ ]:
conv_layer = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=7)
fmaps = conv_layer(cropped_images)

In [ ]:
fmaps.shape

We can visualise all 32 channels of the output feature map for the first image with the following code.

**TODO**: What do you need to change to visualise the result for the second image?

In [ ]:
plt.subplots(figsize=(12, 8))
for i in range(32):
    plt.subplot(4, 8, i + 1)
    plt.imshow(fmaps[0, i].detach().numpy())
    plt.axis("off")

#### Convolution with padding

To use padding within the convolution layer, you can set the `padding` parameter to a specific integer number (0 by default), or specify a string {‘valid’, ‘same’}.

**TODO**: Create a convolutional layer with padding, so that the ouput shape of the feature map is the same as the input.

In [ ]:
# TODO
fmaps = conv_layer(cropped_images)

In [ ]:
fmaps.shape

#### Convolution with stride

To use stride within the convolution layer, you can set the `stride` parameter to a specific integer number (1 by default).

**TODO**: Create a convolutional layer with stride = 2. What will be the shape of the ouput feature map?

In [ ]:
# TODO
fmaps = conv_layer(cropped_images)

In [ ]:
fmaps.shape

#### Weights and biases of a convolutional layer

Let's have a look at the layer's parameters:

In [ ]:
conv_layer.weight.shape

In [ ]:
conv_layer.bias.shape

**TODO**: How many parameters are there in total in this layer?

#### 2. Pooling layer

CNNs are also using another type of layers called *pooling layers*. These layers are used to compress the information in the network. They *subsample* the image to reduce the computational load, memory usage and number of parameters. They can also be seen as selecting relevant information to pass forward in the network. They work similarly to convolutional layers, where each neuron in a layer is connected to a limited number of neurons in the next layer, but they do not filther the image. They calculate a selected statistics (e.g., mean or max) on patches of the input image/feature map and propagate these values forward to the next layer.

The code below shows how to use the `nn.MaxPool2d` layer with a $2 \times 2$ filter. To use avergage poooling, you can use the `nn.AvgPool2d` layer instead.

**TODO**: What should be the shape of the feature map after this max pooling layer?

In [ ]:
max_pool = nn.MaxPool2d(kernel_size=2)
fmaps = max_pool(cropped_images)

In [ ]:
fmaps.shape

Run the following instructions to visualise the output of the max pooling for the sample images. It acts similarly as an image compression tool.

In [ ]:
plt.imshow(fmaps[0].permute(1, 2, 0)) # Need to re-permute the order of the channels for the imshow function

In [ ]:
plt.imshow(fmaps[1].permute(1, 2, 0))

### Putting the layers together: CNN

#### 1. Building a basic CNN

Let's now create a CNN with PyTorch! We can use the Sequential function to stack the layers, but you can also define a class. Bother approaches are shown below.

**Todo**:
Identify the different components of this CNN. How many layers does it have?

*   What types of layers? Do you recognise any other components we discussed in previous tutorials?
*  How did we get the number of input features to the first `Linear` layer?



Notes and tips:
- We first create a default configuration for the convolutional layers we will use (kernel size of 3, same padding, ReLU activation and He normal initialisation), called DefaultConv2D, using the partial function.
- Figuring out the number of input features to the first fully_connected layer can be tricky, especially for larger networks. A tip to find it quickly is to set it to an arbitrary number first (e.g., 999), and let the model crash (e.g., when launching training, or producing a summary). The correct number of features appears in the error message: “RuntimeError: mat1 and mat2 shapes cannot
be multiplied (32x2304 and 999x128)”.
- We use **dropout in the fully-connected layers**, but not in the convolutional and pooling layers. The effect of dropout in convolutional and pooling layers is still not clear and it is not generally advised to use dropout in these types of layers.
- It is usual to **grow the number of filters** with deeper convolutional layers. Indeed, there is usually a small number of low-level features (edges, small circles, ...), but many ways to combine them in higher-level features.
- It is also usual to **double the number of filters** after each pooling layers, as the pooling layer is dividing the spaital dimension by a factor 2 (if the filter size is 2).
- Finally, note that it is common practice to **use small filter size** (typically 3), but you can afford a larger one in the first convolutional layer (maybe even with a stride > 1) to perform a first spatial dimension reduction of the image without loosing too much information.

In [ ]:
# Creates a default configuration for our convolutional layer
DefaultConv2d = partial(nn.Conv2d, kernel_size=3, padding="same")

In [ ]:
# Approach with Sequential function
model = nn.Sequential(
    DefaultConv2d(in_channels=1, out_channels=64, kernel_size=7), nn.ReLU(),
    nn.MaxPool2d(kernel_size=2),
    DefaultConv2d(in_channels=64, out_channels=128), nn.ReLU(),
    DefaultConv2d(in_channels=128, out_channels=128), nn.ReLU(),
    nn.MaxPool2d(kernel_size=2),
    DefaultConv2d(in_channels=128, out_channels=256), nn.ReLU(),
    DefaultConv2d(in_channels=256, out_channels=256), nn.ReLU(),
    nn.MaxPool2d(kernel_size=2),
    nn.Flatten(),
    nn.Linear(in_features=2304, out_features=128), nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(in_features=128, out_features=64), nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(in_features=64, out_features=10),
).to(device)

In [ ]:
# Approach with a class
class Basic_CNN(nn.Module):
    def __init__(self, num_classes=10):
        super(Basic_CNN, self).__init__()
        self.feature_pipeline = nn.Sequential(
            DefaultConv2d(in_channels=1, out_channels=64, kernel_size=7), nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
            DefaultConv2d(in_channels=64, out_channels=128), nn.ReLU(),
            DefaultConv2d(in_channels=128, out_channels=128), nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
            DefaultConv2d(in_channels=128, out_channels=256), nn.ReLU(),
            DefaultConv2d(in_channels=256, out_channels=256), nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)
            )
        self.classification_pipeline = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_features=2304, out_features=128), nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(in_features=128, out_features=64), nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(in_features=64, out_features=10)
            )

    def forward(self, x):
        x = self.feature_pipeline(x)
        x = self.classification_pipeline(x)
        return x

# Need to instantiate a model from the class
# model = Basic_CNN()

After defining the model, you can use the `summary` function from the `torchsummary` library to display a summary of your model.

Note that you need to pass the size of the inputs as well as the model as arguments to the function.

In [ ]:
summary(model, (1,28,28))

#### 2. Training the CNN on MNIST dataset

Let's now train the model on the Fashion MNIST dataset we already used in Lectutorial 2 to train a MLP (we got ~ 0.87 validation accuracy).

We first load, pre-process and split the data, then create data loaders, then define the training function as well as a test set evaluation function, before  launching the training.

In [ ]:
# Load, pre-process, and split the data in train, valid and test sets
toTensor = T.Compose([T.ToImage(), T.ToDtype(torch.float32, scale=True)]) # Define a pre-processing function to convert loaded images to adequate format
train_and_valid_data = torchvision.datasets.FashionMNIST(root="datasets", train=True, download=True, transform=toTensor)
test_data = torchvision.datasets.FashionMNIST(root="datasets", train=False, download=True, transform=toTensor)
train_data, valid_data = torch.utils.data.random_split(train_and_valid_data, [55_000, 5_000])

In [ ]:
# Define the data loaders with a batch size of 32
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
valid_loader = DataLoader(valid_data, batch_size=32)
test_loader = DataLoader(test_data, batch_size=32)

In [ ]:
# Training function returning train and valid losses and evaluation metrics
def train(model, optimizer, loss_fn, eval_metric, train_loader, valid_loader, n_epochs):
    train_losses = []
    train_eval_metrics = []
    valid_losses = []
    valid_eval_metrics= []

    for epoch in range(n_epochs):

        #Training
        eval_metric.reset() # Reset the eval metric
        model.train()
        epoch_train_loss = 0.
        for X_train_batch, y_train_batch in train_loader:
            X_train_batch, y_train_batch = X_train_batch.to(device), y_train_batch.to(device)
            y_train_pred = model(X_train_batch)
            train_loss = loss_fn(y_train_pred, y_train_batch)
            epoch_train_loss += train_loss.item()
            train_loss.backward()
            eval_metric.update(y_train_pred, y_train_batch) # Update eval metric for training
            optimizer.step()
            optimizer.zero_grad()
        mean_epoch_train_loss = epoch_train_loss / len(train_loader)
        train_losses.append(mean_epoch_train_loss)
        # Calculte and store training eval metric for this epoch
        epoch_training_eval_metric = eval_metric.compute().item()
        train_eval_metrics.append(epoch_training_eval_metric)

        # Model evaluation
        model.eval()
        eval_metric.reset() # Reset the eval metric
        epoch_valid_loss = 0.
        with torch.no_grad():
            for X_valid_batch, y_valid_batch in valid_loader:
                X_valid_batch, y_valid_batch = X_valid_batch.to(device), y_valid_batch.to(device)
                y_valid_pred = model(X_valid_batch)
                valid_loss = loss_fn(y_valid_pred, y_valid_batch)
                epoch_valid_loss += valid_loss.item()  # Update eval metric for validation
                eval_metric.update(y_valid_pred, y_valid_batch)
        mean_epoch_valid_loss = epoch_valid_loss / len(valid_loader)
        valid_losses.append(mean_epoch_valid_loss)
        # Calculte and store validation eval metric for this epoch
        epoch_valid_eval_metric = eval_metric.compute().item()
        valid_eval_metrics.append(epoch_valid_eval_metric)

        print(f"Epoch {epoch + 1}/{n_epochs}, Training Loss: {mean_epoch_train_loss:.4f}, Valid Loss: {mean_epoch_valid_loss:.4f}")
        print(f"Epoch {epoch + 1}/{n_epochs}, Training Eval Metric: {epoch_training_eval_metric:.4f}, Valid Eval Metric: {epoch_valid_eval_metric:.4f}")

    return (train_losses, valid_losses, train_eval_metrics, valid_eval_metrics)

In [ ]:
# Function to evaluate the model on the test set (evaluation metric only)
def evaluate_test_set(model, eval_metric, test_loader, metric_print):
    model.eval()
    eval_metric.reset() # Reset the eval metric
    test_loss = 0.
    with torch.no_grad():
      model.eval()
      eval_metric.reset() # Reset the eval metric
      total_test_loss = 0.
      with torch.no_grad():
          for X_test_batch, y_test_batch in test_loader:
              X_test_batch, y_test_batch = X_test_batch.to(device), y_test_batch.to(device)
              y_test_logits = model(X_test_batch)
              y_test_preds = torch.argmax(y_test_logits, dim=1)
              eval_metric.update(y_test_preds, y_test_batch)
      total_test_eval_metric = eval_metric.compute().item()

      print(f"Test {metric_print}: {total_test_eval_metric:.4f}")

In [ ]:
# Define loss (cross entropy), evaluation metric (accuracy), optimiser, number of epochs, and launch the training (takes 6-7 min on Colab T4 GPU)
xentropy = nn.CrossEntropyLoss()
accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=10).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
n_epochs = 20
train_losses, valid_losses, train_accuracy, valid_accuracy = train(model, optimizer, xentropy, accuracy, train_loader, valid_loader, n_epochs)

In [ ]:
# Plot losses and accuracy
plt.plot(train_losses, label='Training loss')
plt.plot(valid_losses, label='Validation loss')
plt.grid()
plt.legend()
plt.show()
plt.plot(train_accuracy, label='Training accuracy')
plt.plot(valid_accuracy, label='Validation accuracy')
plt.grid()
plt.legend()
plt.show()

In [ ]:
# Evalutes the model on the test set
evaluate_test_set(model, accuracy, test_loader, "accuracy")

### 3. Using pre-trained CNNs from TorchVision

In general, you won't need to implement and train your models from scratch. You can load pre-trained weights from well known architectures with only a couple of lines of code using [TorchVision](https://pytorch.org/vision/main/models.html).

The following code loads the weights of a ResNet50 model trained on the ImageNet dataset and creates a model based on the pre-trained weights. `IMAGENET1K_V2` corresponds to the second version of the weights available, you can also use `DEFAULT` to get the best avilable weights.

In [ ]:
weights = torchvision.models.ResNet50_Weights.IMAGENET1K_V2
model = torchvision.models.resnet50(weights=weights).to(device)

You can see the list of all the available weights for a given model with the following instruction.

In [ ]:
list(torchvision.models.get_model_weights("resnet50"))

And the list of all available models with the following instruction.

In [ ]:
torchvision.models.list_models()

To use a model on your own images, you first need to pre-process your images based on the pre-processing used to train the model. This is done with the following instructions.

In [ ]:
transforms = weights.transforms() # Stores information about the pre-processing used in training
preprocessed_images = transforms(sample_images) # Applies this pre-processing to the sample images

Now we can use the model to predict the class of the 2 sample images. The classes are based on the task the model was trained on. Here, it is the ImageNet dataset ([1000 output classes](https://deeplearning.cms.waikato.ac.nz/user-guide/class-maps/IMAGENET/)).

In [ ]:
model.eval()
with torch.no_grad():
    y_logits = model(preprocessed_images.to(device))

In [ ]:
y_logits.shape

The prediction is a $2 \times 1000$ tensor: 1 class logit values per class for each of the 2 images.

We can apply the argmax function to select the predicted class as the one with the highest logit value. Then, we can relate this class to the ImageNet dataset classes names to understand what the predicted class represents.

In [ ]:
y_pred = torch.argmax(y_logits, dim=1)
y_pred

In [ ]:
class_names = weights.meta["categories"]
[class_names[class_id] for class_id in y_pred]

Alternatively, we can display the top 3 predictions and associated probabilities using softmax.

In [ ]:
y_top3_logits, y_top3_class_ids = y_logits.topk(k=3, dim=1)
[[class_names[class_id] for class_id in top3] for top3 in y_top3_class_ids]

In [ ]:
y_top3_logits.softmax(dim=1)

You will see how to **adapt** a pre-trained model to your specific task (e.g., to predict other classes than the ImageNet ones) in the second part of the course.

## Coding time 2: Vision Transformer VS ResNet using Hugging Face

In this last part of the Tutorial, you will use pre-trained versions of the Vision Transformer model and ResNet (CNN based) models to classify images. Both models have been trained on the ImageNet dataset, just as the ResNet-50 model we used in section II.2. We will not go over the implementation of the Vision Transformer architecture, but you can have a look at the following tutorials to do it in PyTorch:
- [UVADCL Tutorial: Vision Transformers (PyTorch)](https://uvadlc-notebooks.readthedocs.io/en/latest/tutorial_notebooks/tutorial15/Vision_Transformer.html)

Some of the code in this part might raise depreciation warnings. If you want to filter them (they are not really useful here), you can run the following cell.

In [ ]:
import warnings
warnings.filterwarnings('error', category=DeprecationWarning)

Next, let's import the `TFViTForImageClassification`, `TFResNetForImageClassification` and `AutoImageProcessor` modules from the  Hugging Face `transformers` package. They contain respective classes and functions to use ViT and ResNet models for classification, and dedicated image processors (to preprocess data for each type of model, similarly as in part II.2. with ResNet-50).

In [ ]:
from transformers import AutoImageProcessor, ViTForImageClassification, ResNetForImageClassification

Next, we will also import the Hugging Face `datasets` package. This package gives you access to the [Hugging Face Datasets library](https://huggingface.co/docs/datasets/en/index), which allows you to easily access and share a range of datasets for Audio, Computer Vision, and Natural Language Processing (NLP) tasks.

In [ ]:
%pip install datasets
from datasets import load_dataset

Let's load an image from the Hugging Face huggingface/cats-image repository to test the models after we load them.

In [ ]:
dataset = load_dataset("huggingface/cats-image")
image = dataset["test"]["image"][0]

In [ ]:
plt.imshow(image)
plt.axis("off")
plt.show()

Let's now load the ViT model and associated processor. The processor can be used to pre-process data to be fed to a specific model.

In [ ]:
model_ViT = ViTForImageClassification.from_pretrained("google/vit-base-patch16-224", torch_dtype=torch.float16)
image_processor_ViT = AutoImageProcessor.from_pretrained("google/vit-base-patch16-224")

In the following code, the image is first preprocessed using the image processor dedicated to this version of the ViT model. Then, a prediction is made with the image we loaded previously. An argmax function is then applied to the output of the model (logits). The class with the highest probability is therefore displayed.

In [ ]:
inputs = image_processor_ViT(image, return_tensors="pt")
logits = model_ViT(**inputs).logits

# model predicts one of the 1000 ImageNet classes
predicted_label = int(torch.argmax(logits, dim=1))
print(model_ViT.config.id2label[predicted_label])

The model classifies the image as in the "Egyptian cat" category, which is one of the 1000 possible classes of the ImageNet dataset. There is indeed a cat on the image (even 2!), but they do not really look like an Egyptian breed. Nervertheless, this is a very good guess out of 1000 classes!

Let see now what ResNet predicts!

We do the exact same process with the pre-trained Resnet-50 model loaded from Hugging Face this time.

In [ ]:
image_processor_ResNet = AutoImageProcessor.from_pretrained("microsoft/resnet-50")
model_ResNet = ResNetForImageClassification.from_pretrained("microsoft/resnet-50")

In [ ]:
inputs = image_processor_ResNet(image, return_tensors="pt")
logits = model_ResNet(**inputs).logits

# model predicts one of the 1000 ImageNet classes
predicted_label = int(torch.argmax(logits, axis=-1))
print(model_ResNet.config.id2label[predicted_label])

The ResNet model classifies the image in the "tiger cat" category. This is also good given the striped pattern exhibited by the cats on the images.

**To continue**: Feel free to play with different part of the code in this tutorial, or just load more images and classify them with the different pre-trained models you loaded in part II.2. and IV.